# ATen 算子的对接与扩展｜学习笔记

> 基于个人原始记录整理。保留原有知识顺序，仅修改明确技术错误、路径、大小写和代码语法；新增可运行的 `torch.library` 最小验证示例放在文末。
> 范围：PyTorch Operator Schema、Kernel、Registration、Dispatcher，以及三种扩展方式。

## 1. 基本定义

- **Operator（算子）**：一个具体计算操作，例如 `add`、`matmul`、`conv2d`、`cat`。
- **Operator Schema（算子接口/契约）**：描述算子名称、参数类型、默认值、返回值、别名与修改语义等。部分形状和设备限制还需要通过实现及额外的元信息规则进行检查，**并非所有约束都直接写在 Schema 字符串里**。
- **Kernel（内核实现）**：具体执行计算的函数或设备代码。
- **Registration（注册）**：将 Schema 和对应的实现登记到 PyTorch 的算子系统。
- **Dispatcher（分发器）**：运行时根据 Dispatch Key 集合与注册情况选择实现。

**ATen** 是 PyTorch 底层的重要 C++ Tensor/算子库，提供 Tensor 相关接口、基础运算与多种后端实现。ATen 并不是单独构成整个 PyTorch 的全部后端架构。

### CPU、CUDA 与其他后端

ATen 算子在不同后端可以有不同实现：CPU 通常是 C++，可使用向量化、多线程及 oneDNN 等优化；CUDA 使用 CUDA C++，也可能调用 cuBLAS、cuDNN 等库。AMD 的 ROCm、Apple 的 MPS 及其他设备后端也可提供对应实现。

最初开发 GPU Kernel 时，**先做正确的 CPU 参考实现，再对比 GPU 输出**，确实是实用的验证方法；但这是一种工程策略，不是注册 GPU Kernel 的先决条件。

**必须纠正：** 缺失 CUDA/其他设备的算子实现时，不会普遍自动将 Tensor 搬到 CPU 执行。可能使用已提供的复合实现或显式 fallback；否则可能提示该后端不支持。手动 CPU 回退还涉及设备拷贝和性能成本。

PyTorch Python API 和 TorchScript 的张量计算可以走到底层算子；TorchServe 是服务部署组件，不应直接等同于 ATen 计算后端。

## 2. 算子的两大机制

**注册（Registration）**：回答“这个名字的算子有哪些可用实现？”。通常先定义 Schema，再将 CPU/CUDA 等 Kernel 注册到 Dispatcher。

**分发（Dispatch）**：回答“这一次调用实际使用哪个实现？”。其 key 不只是设备，还可能涉及 Autograd、Autocast、布局、vmap 等规则和线程局部状态。

**注意区别：** `float32`、`float16` 等 `dtype` 一般不是 CPU/CUDA 那样的 Dispatch Key；通常是在选定的实现内，通过类型检查、模板或 `AT_DISPATCH_*` 宏进一步细分。

```text
调用 torch.ops.my_ops.my_add(x, y)
            |
      Operator Schema
            |
         Dispatcher
      /       |       \
    CPU      CUDA    其他 key/规则
     |         |
 CPU Kernel  CUDA Kernel
```

### C++ 注册示例：`m` 和 `TORCH_FN` 各是什么？

这个例子用自定义命名空间 `my_ops`，**完整展示先定义再注册**。仅是注册片段，`my_cpu_kernel` 和 `my_cuda_kernel` 的真实函数实现需要另外提供。

```cpp
#include <torch/library.h>

TORCH_LIBRARY(my_ops, m) {
    m.def("my_new_op(Tensor x) -> Tensor");
}

TORCH_LIBRARY_IMPL(my_ops, CPU, m) {
    m.impl("my_new_op", TORCH_FN(my_cpu_kernel));
}

TORCH_LIBRARY_IMPL(my_ops, CUDA, m) {
    m.impl("my_new_op", TORCH_FN(my_cuda_kernel));
}
```

- `m` 是宏提供的 **`torch::Library` 对象**，名字本身可换成其他变量名。
- `m.def(...)` 声明 Schema；`m.impl(...)` 登记对应 Kernel。
- `TORCH_FN(my_cpu_kernel)` 是用来包装 C++ 可调用对象的宏，不是 GPU 启动语法，也不是在这里执行 Kernel。对于普通函数，许多情况下可直接写 `&my_cpu_kernel`。
- `aten` 是 PyTorch 的核心算子命名空间；**扩展自定义算子应使用自己命名的命名空间**，不应默认往 `aten` 新增定义。

## 3. 扩展方式一：ATen Native（修改 PyTorch 源码）

原来的流程保留为：确定 Operator Schema → 确定 CPU/CUDA 等实现设备 → 设计高效算法（并行、内存访问、硬件特性）→ 修改 Schema/dispatch 元信息 → 实现 Kernel → 编译与测试。

PyTorch 源码中的位置一般是：

- `aten/src/ATen/native/native_functions.yaml`：声明/配置原生算子的 Schema、变体和 dispatch 等元数据。
- `aten/src/ATen/native/cpu/`：部分 CPU Kernel 的 C++ 实现。
- `aten/src/ATen/native/cuda/`：部分 CUDA Kernel 实现。

源码中的某些注册由 `torchgen` **基于 YAML 生成**，并不要求每个原生算子都手写 `TORCH_LIBRARY_IMPL`。类似 `AT_DISPATCH_FLOATING_TYPES...` 的宏处理实现内部的 `dtype` 模板选择，不等同于 Dispatcher 的设备选择。

**特点：** 深度修改 PyTorch 源码、集成原生构建/发布流程。**不代表天然比 C++ Extension 快**。

## 4. 扩展方式二：`torch.library`（Python 侧算子注册）

`DEF`：定义一个新的算子 Schema。`IMPL`：为已有 Schema 在某个 Dispatch Key 下注册实现，**不是“随便覆盖任何算子”**。`FRAGMENT` 还可在同一命名空间中扩展定义。

下面示例的 CPU 和 CUDA 路径都使用 Python 编写，内部调用 PyTorch 已有运算，仅用于理解注册流程；**它们不是新写的原生 CUDA Kernel**。

正确 Schema 示例：

```python
my_lib = torch.library.Library("learning_ops_demo", "DEF")
my_lib.define("scaled_add(Tensor x, Tensor y, float alpha=1.0) -> Tensor")
```

In [ ]:
import torch

my_lib = torch.library.Library("learning_ops_demo", "DEF")
my_lib.define("scaled_add(Tensor x, Tensor y, float alpha=1.0) -> Tensor")

### 注册 CPU 实现

In [ ]:
def scaled_add_cpu(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    return x + y * alpha

cpu_lib = torch.library.Library("learning_ops_demo", "IMPL", "CPU")
cpu_lib.impl("scaled_add", scaled_add_cpu)

### 注册 CUDA 路径（需要有 CUDA 的环境才能调用验证）

In [ ]:
def scaled_add_cuda(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    return x + y * alpha

cuda_lib = torch.library.Library("learning_ops_demo", "IMPL", "CUDA")
cuda_lib.impl("scaled_add", scaled_add_cuda)

### 验证 CPU / CUDA 输出

此段是额外的最小运行验证。Notebook 重新执行前请重启内核，避免同名 `DEF` 重复注册。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([2.0, 4.0, 6.0])
actual = torch.ops.learning_ops_demo.scaled_add(x, y, 0.5)
expected = x + 0.5 * y
torch.testing.assert_close(actual, expected)
print("CPU:", actual.tolist())

if torch.cuda.is_available():
    a = x.cuda()
    b = y.cuda()
    result_cuda = torch.ops.learning_ops_demo.scaled_add(a, b, 0.5)
    torch.testing.assert_close(result_cuda.cpu(), expected)
    print("CUDA:", result_cuda.cpu().tolist())
else:
    print("CUDA: 当前环境未提供 GPU，未执行 CUDA 测试")

**进一步注意**：真实扩展如需兼容 `torch.compile`、自动求导、混合精度等，还要考虑 FakeTensor/Meta、Autograd、Autocast 等相应注册和测试。官方也提供较高级的 `torch.library.custom_op` API。

## 5. 扩展方式三：PyTorch C++ Extension

主要作用：把高性能 C++/CUDA 实现编译成可加载的扩展，并和 PyTorch 对接。它可以配合 `TORCH_LIBRARY` / `TORCH_LIBRARY_IMPL` / `torch.library` 一起使用。

1. 编写 C++/CUDA 文件（如 `my_extension.cpp`、`my_kernel.cu`）。
2. 为自定义算子定义 Schema、实现 CPU/CUDA Kernel，并选择使用 Dispatcher 注册或 Python 绑定。
3. 使用 `torch.utils.cpp_extension`、setuptools 或 CMake 等配置构建，生成可加载扩展。
4. 从 Python 调用，并做正确性、边界情况、性能、兼容性测试。

```text
my_extension/
├── my_extension.cpp       C++ 包装/注册
├── my_kernel.cu           CUDA 内核
├── setup.py / CMakeLists.txt
└── test_my_op.py          正确性测试
```

FlashAttention、Transformer Engine、vLLM、SGLang 等项目可能使用不同组合的独立扩展、CUDA/Triton Kernel 和框架注册机制，不宜一概断言每一个都只有一种固定实现形式。

## 6. 三种扩展方式总结

| 方法 | 位置 | 主要目的 | 是否要修改 PyTorch 源码 |
|---|---|---|---|
| ATen Native | PyTorch 源码内部 | 为 PyTorch 核心算子开发/维护原生实现 | 通常要 |
| `torch.library` | Python 侧 | 定义 Schema、注册 Kernel/子系统规则 | 不需要 |
| C++ Extension | 独立编译的 C++/CUDA 扩展 | 集成高性能原生 Kernel | 不需要 |

**一句话：** Schema 规定“调用什么”，Kernel 执行“怎么计算”，Registration 说明“有哪些实现”，Dispatcher 决定“这次调用哪个实现”。

不要把 **Backend Dispatch** 和 **dtype 模板分发**混为一谈，也不要认为缺少 GPU Kernel 时会自动回落 CPU。

## 7. 官方参考资料（进一步核对）

- [PyTorch Custom Operators](https://docs.pytorch.org/tutorials/advanced/custom_ops_landing_page.html)
- [Custom C++ and CUDA Operators](https://docs.pytorch.org/tutorials/advanced/cpp_custom_ops.html)
- [`torch.library` API](https://docs.pytorch.org/docs/stable/library.html)
- [C++ Library registration](https://docs.pytorch.org/cppdocs/api/library/registration.html)
- [PyTorch Codegen and Structured Kernels](https://github.com/pytorch/pytorch/wiki/Codegen-and-Structured-Kernels)

记录日期：2026-10-27。当前笔记中的 Python 示例在可用的 PyTorch 环境测试；GPU 运行依环境是否有 CUDA 决定。C++ 注册片段只做静态说明，尚未独立编译。